In [ ]:
import qsharp
import numpy as np
print(f"qsharp version: {__import__("importlib.metadata", fromlist=["version"]).version("qsharp")}")

<h3>A pure function — no qubits, no side effects</h3>

In [ ]:
%%qsharp

// A pure function — no qubits, no side effects
function Square(x : Int) : Int {
    ???
}

// An operation — allocates qubits, applies gates
operation BellState() : Result[] {
    use q = Qubit[2];
    H(q[0]);
    CNOT(q[0], q[1]);
    ???
}

// Lambda: classical function lambda
let double = x -> x * 2;

// Partial application: fix the control qubit, leaving target open
// (used later inside an operation)

Message($"Square(7) = {Square(7)}");
Message($"double(5) = {double(5)}");
let result = BellState();
Message($"BellState result: {result}");

<h3>Allocate 3 qubits — scoped to this block</h3>

In [ ]:
%%qsharp

import Std.Diagnostics.DumpMachine;

operation ScopeDemo() : Unit {
    // Allocate 3 qubits — scoped to this block
    use qs = Qubit[3];
    H(qs[0]);
    ???
    CNOT(qs[0], qs[2]);
    Message("GHZ state:");
    DumpMachine();
    ResetAll(qs);  // mandatory before release
}

ScopeDemo()

<h3>Marking is Adj + Ctl lets the compiler derive both functors</h3>

In [ ]:
%%qsharp

import Std.Diagnostics.DumpMachine;

// Marking is Adj + Ctl lets the compiler derive both functors
operation PrepareState(q : Qubit) : Unit is Adj + Ctl {
    ???
    H(q);
    // State is now |−⟩ = (|0⟩ - |1⟩)/√2
}

operation FunctorDemo() : Unit {
    use q = Qubit();
    PrepareState(q);
    Message("After PrepareState:");
    DumpMachine();

    // Adjoint uncomputes the state back to |0⟩
    Adjoint PrepareState(q);
    Message("After Adjoint PrepareState (back to |0⟩):");
    DumpMachine();

    // within/apply: conjugate pattern
    within {
        H(q);  // change basis
    } apply {
        Z(q);  // apply Z in X-basis = apply X in Z-basis
    }
    Message("After within/apply (H Z H = X):");
    DumpMachine();
    Reset(q);
}

FunctorDemo()

<h3>Arrays: functional-style combinators</h3>

In [ ]:
%%qsharp

import Std.Arrays.*;
import Std.Math.*;
import Std.Diagnostics.DumpMachine;
import Std.StatePreparation.PreparePureStateD;

// Arrays: functional-style combinators
let nums = [1, 2, 3, 4, 5];
let squares = ???
let evens = Filtered(x -> x % 2 == 0, nums);
Message($"Squares: {squares}");
Message($"Evens: {evens}");
Message($"Head: {Head(nums)}, Tail: {Tail(nums)}");

// Math
Message($"PI = {PI()}");
Message($"Sqrt(2) = {Sqrt(2.0)}");

// StatePreparation: prepare arbitrary superposition
operation PrepDemo() : Unit {
    use qs = Qubit[3];
    // Uniform superposition over first 4 basis states
    let coeffs = [0.5, 0.5, 0.5, 0.5, 0.0, 0.0, 0.0, 0.0];
    PreparePureStateD(coeffs, qs);
    Message("Uniform superposition over |000⟩..|011⟩:");
    DumpMachine();
    ResetAll(qs);
}

PrepDemo()

<h3>qsharp.eval — single shot, returns Python value</h3>

In [ ]:
# qsharp.eval — single shot, returns Python value
result = qsharp.eval("""
    function AddTwoInts(a : Int, b : Int) : Int {
        return a + b;
    }
    AddTwoInts(40, 2)
""")
???

<h3>RandomBit operation</h3>

In [ ]:
%%qsharp

operation RandomBit() : Result {
    use q = Qubit();
    H(q);
    ???
}

<h3>qsharp.run — multiple shots</h3>

In [ ]:
# qsharp.run — multiple shots
results = qsharp.run("RandomBit()", 20)
from collections import Counter
counts = ???
print(f"Results from 20 shots: {dict(counts)}")

<h3>qsharp.code.* — direct Python callable</h3>

In [ ]:
# qsharp.code.* — direct Python callable
from qsharp.code import AddTwoInts
???

from qsharp.code import RandomBit
print(f"RandomBit() = {RandomBit()}")

<h3>Error handling</h3>

In [ ]:
# Error handling
from qsharp import QSharpError

try:
    qsharp.eval("NonExistentOperation()")
except QSharpError as e:
    ???

<h3>RandomNBits inside a named namespace</h3>

In [ ]:
%%qsharp

namespace Course.Utils {
    operation RandomNBits(n : Int) : Result[] {
        use qs = Qubit[n];
        ???
        MResetEachZ(qs)
    }
}

<h3>Call namespace operation from Python</h3>

In [ ]:
from qsharp.code.Course.Utils import RandomNBits
???
print(f"8 random bits: {bits}")

<h3>Simulate and collect 10 shots</h3>

In [ ]:
from qsharp.openqasm import run, compile, import_openqasm

bell_qasm = """
    include "stdgates.inc";
    bit[2] c;
    qubit[2] q;
    h q[0];
    cx q[0], q[1];
    c = measure q;
"""

# Simulate and collect 10 shots
shots = ???
print("Bell state shots:", shots)

<h3>Compile to QIR</h3>

In [ ]:
# Compile to QIR
compilation = ???
print("QIR (first 500 chars):\n", str(compilation)[:500])

<h3>Import as a named Q# callable for interactive use</h3>

In [ ]:
# Import as a named Q# callable for interactive use
qsharp.init(target_profile=qsharp.TargetProfile.Base)
???

from qsharp.code import bell_from_qasm
print("Bell callable result:", bell_from_qasm())

<h3>Conceptual — requires qiskit and qiskit-qsharp to be installed</h3>

In [ ]:
# Conceptual — requires qiskit and qiskit-qsharp to be installed
try:
    from qiskit import QuantumCircuit
    from qsharp.interop.qiskit import QSharpBackend

    qc = QuantumCircuit(2, 2)
    ???
    qc.cx(0, 1)
    qc.measure([0, 1], [0, 1])

    backend = QSharpBackend()
    job = backend.run(qc, shots=100)
    counts = job.result().get_counts()
    print("Qiskit via QSharpBackend:", counts)
except ImportError:
    print("qiskit not installed — skipping. Install with: pip install qiskit qiskit-qsharp")